# Exploring the raw Piura land/crop datasets

Two source files sit in `data/raw/`:

- `BD SSET(MOQUEGUA-PASCO-PIURA).xlsx` — a flat table covering Moquegua, Pasco and Piura.
- `Piura.dta` — a Stata file with 65 columns, Piura only.

The goal of this notebook is purely diagnostic: load each file correctly, describe it,
figure out what it actually is (as opposed to what we assumed going in), and work out
what else we'd need to turn them into (polygon, crop, date) → satellite image training
examples for a crop classifier.

In [1]:
import socket
socket.setdefaulttimeout(90)  # global cap so no tile/GEE fetch can hang the run

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import contextily as ctx
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RAW = Path("../data/raw")
XLSX_PATH = RAW / "BD SSET(MOQUEGUA-PASCO-PIURA).xlsx"
DTA_PATH = RAW / "grafica_tabular_catastro_Piura.dta"
DTA_PATH = RAW / "grafica_tabular_Piura.dta"


## 1. `BD SSET(MOQUEGUA-PASCO-PIURA).xlsx`

Single sheet, `DATOS`, ~510k rows. Loaded directly with `pandas.read_excel` (via the
`openpyxl` engine, added as a dependency). Dates and numeric-looking columns are coerced
explicitly since Excel mixes text and native types in these columns.

In [2]:
sset = pd.read_excel(XLSX_PATH, sheet_name="DATOS", engine="openpyxl")
sset.columns = [c.strip() for c in sset.columns]

sset["FECHA EMPADRONAMIENTO"] = pd.to_datetime(sset["FECHA EMPADRONAMIENTO"], errors="coerce")
sset["AREA"] = pd.to_numeric(sset["AREA"], errors="coerce")
for c in ["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "CULTIVO", "ESTADO en RRPP"]:
    sset[c] = sset[c].astype("string").str.strip().str.upper()

print(sset.shape)
sset.head()

(509684, 13)


,DEPARTAMENTO,PROVINCIA,DISTRITO,NOMBRES,DNI,FECHA EMPADRONAMIENTO,ESTADO en RRPP,AREA,CULTIVO,Codigo SSET,Codigo Proyecto,DIAS_11,DIAS_12
0,MOQUEGUA,MARISCAL NIETO,MOQUEGUA,VALDIVIA CASTILLO MARIA LUCILA,04411958,1996-01-01,NO INSCRITO,31233.0,ALFALFA,210001566,056B,5843,6209
1,MOQUEGUA,GENERAL SANCHEZ CERRO,OMATE,AYALA TODCO AURORA,04720939,1996-01-01,INSCRITO,886.0,ALFALFA,210014902,060,5843,6209
2,MOQUEGUA,GENERAL SANCHEZ CERRO,OMATE,CHICHIZOLA PORRAS CESAR AUGUSTO,29657075,1996-01-01,INSCRITO,2361.0,ALFALFA,210017573,060,5843,6209
3,MOQUEGUA,GENERAL SANCHEZ CERRO,PUQUINA,PERALTILLA DELGADO OCTAVIO WALBERTO,04728963,1996-01-01,INSCRITO,2893.0,ALFALFA,210008073,059,5843,6209
4,MOQUEGUA,GENERAL SANCHEZ CERRO,PUQUINA,RODRIGUEZ YAGUA HERMELINDA BALVINA,29406132,1996-01-01,INSCRITO,2893.0,ALFALFA,210008073,059,5843,6209


In [3]:
sset.info()

<class 'pandas.DataFrame'>
RangeIndex: 509684 entries, 0 to 509683
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   DEPARTAMENTO           509678 non-null  string        
 1   PROVINCIA              509678 non-null  string        
 2   DISTRITO               509678 non-null  string        
 3   NOMBRES                492962 non-null  str           
 4   DNI                    421041 non-null  str           
 5   FECHA EMPADRONAMIENTO  509684 non-null  datetime64[us]
 6   ESTADO en RRPP         509684 non-null  string        
 7   AREA                   478687 non-null  float64       
 8   CULTIVO                296274 non-null  string        
 9   Codigo SSET            509684 non-null  int64         
 10  Codigo Proyecto        509684 non-null  str           
 11  DIAS_11                509684 non-null  int64         
 12  DIAS_12                509684 non-null  int64         


In [4]:
# Missingness by column
sset.isna().mean().sort_values(ascending=False).to_frame("frac_missing")

,frac_missing
CULTIVO,0.418710
DNI,0.173918
AREA,0.060816
NOMBRES,0.032809
DEPARTAMENTO,0.000012
PROVINCIA,0.000012
DISTRITO,0.000012
FECHA EMPADRONAMIENTO,0.000000
ESTADO en RRPP,0.000000
Codigo SSET,0.000000


In [5]:
sset["DEPARTAMENTO"].value_counts(dropna=False)

DEPARTAMENTO
PIURA       348076
MOQUEGUA     95021
PASCO        66528
AYACUCHO        49
<NA>             6
UCAYALI          4
Name: count, dtype: int64[pyarrow]

In [6]:
sset["ESTADO en RRPP"].value_counts(dropna=False)

ESTADO en RRPP
NO INSCRITO    352291
INSCRITO       157393
Name: count, dtype: int64[pyarrow]

In [7]:
# Registration-date coverage: is this a single census wave or a long-running register?
year_counts = sset["FECHA EMPADRONAMIENTO"].dt.year.value_counts().sort_index()
ax = year_counts.plot(kind="bar", figsize=(10, 3), title="FECHA EMPADRONAMIENTO by year")
ax.set_ylabel("records")
plt.tight_layout()
plt.show()

print("Date range:", sset["FECHA EMPADRONAMIENTO"].min(), "->", sset["FECHA EMPADRONAMIENTO"].max())
print()
print("Most common exact dates (look for stub/batch dates):")
print(sset["FECHA EMPADRONAMIENTO"].value_counts().head(8))

<Figure size 1000x300 with 1 Axes>

Date range: 1996-01-01 00:00:00 -> 2019-08-05 00:00:00

Most common exact dates (look for stub/batch dates):
FECHA EMPADRONAMIENTO
2001-01-01    175744
1999-04-04      8199
1998-04-04      5745
1997-04-04      1996
1998-09-01      1637
1998-10-01      1542
1998-10-20      1351
2002-01-01      1296
Name: count, dtype: int64


The single date `2001-01-01` alone accounts for ~176k of ~510k rows, and several other
dates (`1999-04-04`, `1998-04-04`, ...) each account for thousands of rows. Real registration
events don't cluster like that — these look like **stub/placeholder dates** stamped on batches
of records when they were digitised or migrated between systems, not the true date a farmer
was surveyed. `FECHA EMPADRONAMIENTO` is safe to use for coarse eligibility/coverage checks but
should not be trusted as a precise "date of observation" for a given parcel.

In [8]:
piura = sset[sset["DEPARTAMENTO"] == "PIURA"].copy()
print(f"Piura rows: {len(piura):,} of {len(sset):,}")

piura["CULTIVO"].value_counts(dropna=False).head(20)

Piura rows: 348,076 of 509,684


CULTIVO
<NA>                   170381
ARROZ                   53795
MAIZ                    17404
EN DESCANSO             10154
ALGODON                  7539
GRAMA CRIOLLA            5224
DESCANSO                 5063
TERRENO EN DESCANSO      4662
PASTO ELEFANTE           3418
TRIGO                    3155
MANGO                    2842
PLATANO                  2012
PASTO                    1840
LIMON                    1823
PAJA CHILENA             1395
FREJOL                   1330
CAÑA DE AZUCAR           1083
GRAMA CHILENA            1058
CAFE                      938
CAFE Y PLATANO            820
Name: count, dtype: int64[pyarrow]

In [9]:
piura["AREA"].describe()

count    3.355250e+05
mean     1.870887e+04
std      1.154203e+06
min      0.000000e+00
25%      1.045000e+03
50%      3.559000e+03
75%      9.517000e+03
max      3.901481e+08
Name: AREA, dtype: float64

`AREA`'s median (~3,200) and IQR are consistent with **square metres**, not hectares — a
3,200 ha parcel would be enormous, but 0.32 ha is a plausible smallholder plot. There are also
negative values and a max of ~3.9e8, which are clearly data errors and will need to be filtered
before this column is used for anything (e.g. as a plausibility check on parcel size, or a
bounding-box radius for imagery extraction).

In [10]:
# How many crop records exist per parcel? (Codigo SSET is the parcel/farmer key)
per_code = piura["Codigo SSET"].value_counts()
print(per_code.describe())
per_code.value_counts().sort_index().head(10).rename("n_parcels_with_k_records")

count    209485.000000
mean          1.661580
std           0.695158
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          27.000000
Name: count, dtype: float64


count
1      83606
2     120406
3        590
4       4100
5         68
6        466
7         59
8         86
9         29
10        24
Name: n_parcels_with_k_records, dtype: int64

Most parcels have 1–2 crop records, but some have up to 27. So this table is really a
**panel**: the same `Codigo SSET` parcel reappears across years with a (possibly) different
`CULTIVO` and `FECHA EMPADRONAMIENTO` each time — i.e. a crop history, not one row per farm.

### What this dataset likely is

This is **not** a population census. It reads as an administrative **land-titling / crop
registry** (SSET = "Sistema de Seguimiento..." tracking system tied to Peru's PETT land-titling
programme), with one row per parcel-crop declaration: who farms it (`NOMBRES`, `DNI`), where
(`DEPARTAMENTO`/`PROVINCIA`/`DISTRITO`), what's growing (`CULTIVO`), how large (`AREA`), and its
titling/registration status (`ESTADO en RRPP`, `FECHA EMPADRONAMIENTO`). It spans 1996–2019 and
three departments (Piura dominates at ~348k of ~510k rows), which matches the user's "PETT
register 1994–2019" hypothesis reasonably well — the earliest usable dates start in 1996-97
rather than 1994, and the bulk of "dates" are unreliable stub values as shown above.

## 2. `Piura.dta`

109,796 rows, 65 columns. This is a Stata export (release 118 / Stata 18 format), read here
with `pyreadstat` via `pandas.read_stata`-equivalent for robustness. A number of the columns
(`dup`, `dup2`, `dup3`, `_merge`, `merge_t_g_c`) are clearly leftovers from a prior Stata
`merge` operation and are dropped for readability.

In [11]:
piura_dta = pd.read_stata(DTA_PATH, convert_categoricals=False)
junk_cols = [c for c in piura_dta.columns if c in
             {"dup", "dup2", "dup3", "_merge", "merge_t_g_c"}]
piura_dta = piura_dta.drop(columns=junk_cols)
print(piura_dta.shape)
piura_dta.head()

(158427, 46)


,objectid,COD_PREDIO,hoja,escala,area_ha,perimetro,centroid_e,centroid_n,DEPARTAMENTO,PROVINCIA,DISTRITO,sector,caserio,valle,predio,condicion,NOMBRES,observacio,num_predio,origen,datum,estado,calculo,id_dist,id_prov,id_dpto,proy,antecedent,id_gdb,CodigoSSET,id_pred_ru,c_est_exp,c_est_rrpp,cod_estado,cod_est_ru,c_codofic,c_idvuelo,cod_uso,id_proy_ca,fech_tran,codi_user,id_plano,st_area_sh,st_length_,DIAS_MAX_11,DIAS_MAX_12
0,1545486,7_6359485_001878,7_6359485,1/10000,11.4667,1555.02,639501,9487020,PIURA,AYABACA,AYABACA,MOLINOS DE SANGULI,,QUIROZ,,PROPIETARIO UNICO,FLORES TOCTO JESUS MARINA,Una persona vinculada,001878,GPS,PSAD56,PREDIO INSCRITO ANTES DEL PETT,CARTESIANO,200201,2002,20,P109,NaN,X02,230190729,PRU-00000005422577,3,,EG003,EX003,NaN,109,NaN,,2011-05-20,RMAZA,,9.347984e-06,0.014044,1967.0,2333.0
1,2403939,7_6359485_001876,7_6359485,1/10000,10.5943,1601.64,638901,9486840,PIURA,AYABACA,AYABACA,MOLINOS DE SANGULI,,QUIROZ,,PROPIETARIO UNICO,FLORES TOCTO ANDREA,Una persona vinculada,001876,GPS,PSAD56,PREDIO INSCRITO ANTES DEL PETT,CARTESIANO,200201,2002,20,P109,NaN,X02,230190732,PRU-00000005428301,3,,EG003,EX003,NaN,109,NaN,,2011-05-20,RMAZA,,8.636813e-06,0.014464,1967.0,2333.0
2,386196,7_6359485_001874,7_6359485,1/10000,10.3544,1368.27,639170,9487268,PIURA,AYABACA,AYABACA,MOLINOS DE SANGULI,,QUIROZ,,COPROPIETARIO,FLORES GUERRERO ALEJANDRO,"Varias personas vinculadas, ver Sistema SSET",001874,GPS,PSAD56,PREDIO INSCRITO ANTES DEL PETT,CARTESIANO,200201,2002,20,P109,NaN,X02,230190733,PRU-00000005425464,3,,EG003,EX003,NaN,109,NaN,,2011-05-20,RMAZA,,8.441219e-06,0.012357,1967.0,2333.0
3,1521869,7_6359485_001871,7_6359485_2_4,1/2500,0.4960,399.15,638659,9487173,PIURA,AYABACA,AYABACA,MOLINOS DE SANGULI,,QUIROZ,,COPOSEEDOR,CASTILLO SAAVEDRA DE FLORES DORILA,"Varias personas vinculadas, ver Sistema SSET",001871,GPS,PSAD56,PREDIO INSCRITO ANTES DEL PETT,CARTESIANO,200201,2002,20,P109,NaN,X02,230190736,PRU-00000005429457,3,,EG003,EX003,NaN,109,NaN,,2011-05-20,RMAZA,,4.043296e-07,0.003602,1967.0,2333.0
4,1364997,7_6359485_001868,7_6359485_2_4,1/2500,1.2955,529.83,638670,9486738,PIURA,AYABACA,AYABACA,MOLINOS DE SANGULI,,QUIROZ,,COPROPIETARIO,FLORES GUERRERO OSCAR ERIBERTO,"Varias personas vinculadas, ver Sistema SSET",001868,GPS,PSAD56,PREDIO INSCRITO ANTES DEL PETT,CARTESIANO,200201,2002,20,P109,NaN,X02,230190738,PRU-00000005426386,3,,EG003,EX003,NaN,109,NaN,,2011-05-20,RMAZA,,1.056139e-06,0.004784,1967.0,2333.0


In [12]:
piura_dta.isna().mean().sort_values(ascending=False).head(20).to_frame("frac_missing")

,frac_missing
c_codofic,1.000000
cod_uso,1.000000
antecedent,1.000000
DIAS_MAX_12,0.000271
DIAS_MAX_11,0.000271
fech_tran,0.000006
st_area_sh,0.000000
c_est_rrpp,0.000000
id_dpto,0.000000
proy,0.000000


In [13]:
piura_dta["area_ha"].describe()

count    158427.000000
mean          1.882361
std         192.865523
min           0.001000
25%           0.195000
50%           0.473100
75%           1.112400
max       76443.949100
Name: area_ha, dtype: float64

In [14]:
piura_dta["condicion"].value_counts(dropna=False)

condicion
SIN COND JURIDCA        50377
COPOSEEDOR              42968
POSEEDOR                37804
PROPIETARIO UNICO       18253
COPROPIETARIO            7787
                          583
PRECARIO                  506
SOCIEDAD CONYUGAL          49
COTITULARIDAD              46
PROPIETARIO                40
SOCIEDAD INTESTADA          5
POSESIONARIO                4
HEROS DE TRONCOSO, M        1
CHUMPITAZ PANTA, MOI        1
EX CAT SINCHI               1
OLIVARES CARMEN, LOR        1
ARISMENDIS OJEDA, AR        1
Name: count, dtype: int64

In [15]:
piura_dta["estado"].value_counts(dropna=False).head(10)

estado
Exp: EN PROCESO (CONFORMACI├ôN DE EXPEDIENTE)         47710
PREDIO INSCRITO ANTES DEL PETT                        42295
RRPP: ENVIADO                                         23848
RRPP: PROPIEDAD INSCRITA                              20053
Exp: OBSERVADO                                         9701
Exp: COMPLETO (APTO PARA LA PRESENTACI├ôN A SUNARP     9237
RRPP: PUBLICACION EN EL PERUANO                        1373
RRPP: PRESENTADO                                       1213
No hay Expediente en el SSET                            966
Exp: PENDIENTE                                          934
Name: count, dtype: int64

In [16]:
for c in ["datum", "origen", "escala"]:
    print(piura_dta[c].value_counts(dropna=False))
    print()

datum
PSAD56    158427
Name: count, dtype: int64

origen
FOTOGRAMETRICO    149349
GPS                 9078
Name: count, dtype: int64

escala
1/2500      142012
1/5000        9714
1/10000       6463
1/100000       232
2500             6
Name: count, dtype: int64



`datum` is uniformly `PSAD56` and the `centroid_e`/`centroid_n` values (easting
~486k–684k, northing ~9.35M–9.55M) are UTM-scale coordinates in the southern hemisphere —
consistent with **PSAD56 / UTM zone 17S (EPSG:24877)**, which covers Piura. We build point
geometries from the centroids and reproject to WGS84 (EPSG:4326) to sanity-check the
locations and make the data usable with GEE later (GEE expects WGS84 lon/lat).

In [17]:
geom = gpd.points_from_xy(piura_dta["centroid_e"], piura_dta["centroid_n"])
parcels_pts = gpd.GeoDataFrame(piura_dta, geometry=geom, crs="EPSG:24877").to_crs(4326)

# Plain scatter (no basemap): this is just a sanity check that the centroids fall inside
# Piura's bounding box. A department-wide satellite tile fetch here repeatedly stalled the
# run, and it adds no analytical value at this scale -- the satellite-overlaid, parcel-level
# views are in sections 5, 7 and 8. (Set fetch_basemap=True to re-enable a low-zoom overlay.)
fetch_basemap = False

fig, ax = plt.subplots(figsize=(8, 8))
parcels_pts.plot(ax=ax, markersize=1, color="red", alpha=0.15)
if fetch_basemap:
    import socket; socket.setdefaulttimeout(30)
    try:
        ctx.add_basemap(ax, crs=parcels_pts.crs, source=ctx.providers.Esri.WorldImagery,
                         zoom=8, timeout=30)
    except Exception as e:
        print(f"[basemap skipped: {type(e).__name__}]")
ax.set_title(f"Parcel centroids (n={len(parcels_pts):,}) -- Piura bounding-box sanity check")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.tight_layout(); plt.show()

print(parcels_pts.geometry.x.describe())
print(parcels_pts.geometry.y.describe())


<Figure size 800x800 with 1 Axes>

count    158427.000000
mean        -80.361996
std           0.556413
min        -171.220947
25%         -80.734743
50%         -80.639367
75%         -79.964901
max         -79.338370
dtype: float64
count    158427.000000
mean         -5.213478
std           0.329790
min         -85.524199
25%          -5.408606
50%          -5.290703
75%          -4.941965
max          -4.092753
dtype: float64


The point cloud falls inside Piura department's bounding box (~-81 to -79.5 lon,
-6.4 to -4 lat), which checks out.

### What this dataset likely is

Given the cadastral fields (`centroid_e/n`, `area_ha`, `perimetro`, `escala`, `origen` =
`FOTOGRAMETRICO`/`GPS`, `datum`), the PETT titling-status fields (`condicion`, `estado`,
`cod_est_ru`), and the very narrow `fech_tran` window (2011-05-20 to 2011-06-15 — almost
certainly an export/transaction timestamp rather than a survey date), this looks like the
**cadastral survey base table for Piura's PETT parcels, produced around 2011** — most likely
the geodetic layer used to build the parcel frame for the 2012 agricultural census (CENAGRO),
rather than the census microdata itself. The stray `P001`/`P002`/`P003`/`P009_01-03` columns
(question-code-style names, holding name/surname fragments) support a link to a census-style
questionnaire, but this table itself carries **no crop information at all** (`cod_uso` is
100% missing) — for crop labels we need the SSET workbook above.

Crucially, it holds **no polygon geometry** — only a centroid, an area, and a perimeter per
parcel. The real parcel polygons must live in whatever shapefile/geodatabase these attributes
were originally exported from; that source isn't in `data/raw/` yet.

## 3. Linking the two tables

Both tables carry a shared code: `Codigo SSET` in the workbook and `CodigoSSET` in the Stata
file. If they link cleanly, we can attach a real (approximate) location to each crop
declaration.

In [18]:
dta_codes = set(piura_dta["CodigoSSET"].dropna().astype(str))
xlsx_codes = set(piura["Codigo SSET"].dropna().astype(str))

overlap = dta_codes & xlsx_codes
print(f"Unique parcel codes in Piura.dta:        {len(dta_codes):,}")
print(f"Unique parcel codes in xlsx (Piura rows): {len(xlsx_codes):,}")
print(f"Overlap:                                  {len(overlap):,}  "
      f"({len(overlap) / len(dta_codes):.1%} of Piura.dta parcels matched)")

Unique parcel codes in Piura.dta:        157,857
Unique parcel codes in xlsx (Piura rows): 209,485
Overlap:                                  157,580  (99.8% of Piura.dta parcels matched)


~99.8% of the cadastral parcels in `Piura.dta` have at least one matching crop record in
the SSET workbook — this join key is reliable and is the backbone for anything downstream.

## 4. Path to classifier training data — what we have, what's missing

**Target shape:** for each farm, a polygon + a crop label + a date → pull the matching
satellite image → train.

**What these two tables give us, once joined on `Codigo SSET` / `CodigoSSET`:**
- Crop label (`CULTIVO`) and a registration date, from the SSET workbook, per parcel-year.
- A location (centroid) and an area, from `Piura.dta`, per parcel — reprojectable to WGS84
  for GEE.
- A near-complete (~99.8%) join between the two.

**Gaps that need closing before this becomes training data:**

1. **No polygons.** We only have a centroid + area + perimeter, not parcel boundaries. Either
   (a) locate the original shapefile/geodatabase these cadastral attributes came from — the
   real fix — or (b) approximate an extent as a circle/square sized from `area_ha` around the
   centroid, clearly labelled as an approximation, for an initial pass.
2. **Unreliable dates.** `FECHA EMPADRONAMIENTO` clusters heavily on stub batch dates
   (`2001-01-01` alone is ~35% of rows) rather than true survey/planting dates. Using it
   directly to pick a single satellite scene risks pulling an image from the wrong season
   entirely. A phenology-aware window (e.g. the known growing season for the declared crop in
   this region) is probably safer than trusting the exact date.
3. **Crop-label cleanup.** `CULTIVO` needs normalisation before it's a classifier target:
   distinguishing real crops from fallow (`DESCANSO`, `EN DESCANSO`, `TERRENO EN DESCANSO`),
   handling missing values (~49% missing in the raw Piura crop column), and deciding how to
   handle intercropped/mixed entries (e.g. `CAFE Y PLATANO`) — drop, keep as their own class,
   or split into multi-label.
4. **Area outliers.** `AREA` in the workbook has negative values and implausible maxima; needs
   filtering (and unit reconciliation — it looks like m² in the workbook vs. `area_ha` already
   in hectares in the Stata file) before it's used for anything spatial.
5. **Panel vs. single-row.** Most parcels have 1–2 crop records but some have up to 27 — decide
   whether to model this as one row per parcel-year (recommended, since crop and season change)
   rather than collapsing to one row per parcel.

**Proposed pipeline once the above is resolved:**
1. Clean & join: normalise `CULTIVO`, drop/flag fallow, inner-join the Piura SSET rows to
   `Piura.dta` on the SSET code to attach geometry to each crop-year record.
2. Build a per-record extent in WGS84 (true polygon if the source shapefile can be found;
   otherwise the area-derived approximation from point 1 above).
3. For each parcel-year record, query GEE (Sentinel-2/Landsat, cloud-masked) over a
   season-aware window, clipped to the parcel extent, and extract summary features (band
   medians, NDVI/EVI, etc.).
4. Split train/test **by parcel or farmer id**, not by row — several rows share the same
   parcel across years, and letting them land on both sides of the split would leak.
5. Train a baseline classifier (e.g. Random Forest / gradient boosting) on the extracted
   features to predict `CULTIVO`.

**Open questions worth resolving with the domain side before building the pipeline:**
- Is there a source shapefile/geodatabase with the real parcel polygons behind `Piura.dta`?
- What planting-to-harvest window should we assume per crop, given the registration date is
  unreliable?
- How should mixed-crop entries be handled — separate class, multi-label, or dropped?

## 5. The CENAGRO cadastral polygons — `data/raw/qgis_stefany/`

This folder is the missing piece flagged in section 4: it contains an **Esri shapefile**
(`CATASTRO_CENAGRO_PIURA_WGS84_Z17S_FINAL.*`) with the actual **parcel boundary polygons**,
plus a companion `PIURA.dta`.

A shapefile is a multi-file format; all of these must travel together:
- `.shp` — the geometry (the polygons themselves)
- `.shx` — geometry index
- `.dbf` — attribute table (one row per polygon)
- `.prj` — coordinate reference system
- `.cpg` — attribute text encoding; `.qix` — spatial index; `.shp.xml` — ESRI metadata

`geopandas.read_file` reads the whole set from the `.shp` path. The `.prj` declares
**WGS84 / UTM zone 17S (EPSG:32717)** — the modern WGS84 equivalent of the PSAD56/UTM-17S
frame the centroid table used, so these polygons sit in the same place as section 2's points.

In [19]:
SHP_PATH = RAW / "qgis_stefany" / "CATASTRO_CENAGRO_PIURA_WGS84_Z17S_FINAL.shp"

# Read attributes only first (fast) to understand the schema before loading 190k polygons.
cad_attrs = gpd.read_file(SHP_PATH, ignore_geometry=True)
print("features:", len(cad_attrs))
print("columns:", list(cad_attrs.columns))
cad_attrs.head()

features: 190098
columns: ['COD_INEI', 'UBIGEO', 'NOMBRE', 'NUM_PREDIO', 'AREA_S_HA', 'N_SEA', 'COND_JUR', 'COD_PREDIO', 'FUENTE', 'dentro1km']


,COD_INEI,UBIGEO,NOMBRE,NUM_PREDIO,AREA_S_HA,N_SEA,COND_JUR,COD_PREDIO,FUENTE,dentro1km
0,None,200401,AGRICOLA EL MILAGRO S.A.C.,088934,22.249829,None,2,7_5859435_088934,1,NaN
1,None,200401,"RIVAS GA, MARIA ROMELIA",088991,10.463520,None,1,7_5859445_088991,1,NaN
2,None,200111,"OROZCO GARCIA, FREDES",32680,0.136596,None,1,7_6009480_32680,1,NaN
3,None,200301,"MAURIOLA GONZALES, AFRANIO",72019,3.237167,None,1,7_6709420_72019,1,NaN
4,None,200301,"NEYRA HUAYAMA, JOSE ROSARIO",70671,0.086017,None,1,7_6659425_70671,1,NaN


**Column glossary** (INEI/CENAGRO cadastral conventions):

| column | meaning |
|---|---|
| `COD_PREDIO` | parcel code, e.g. `7_5859435_088934` — **the join key** to the cadastral `.dta` |
| `UBIGEO` | 6-digit district code (department 20 = Piura) |
| `NOMBRE` | holder / farm name |
| `NUM_PREDIO` | parcel number (the tail of `COD_PREDIO`) |
| `AREA_S_HA` | polygon area in hectares (computed from the shape) |
| `COND_JUR` | juridical condition code (1/2/3) |
| `FUENTE` | source/provenance code for the geometry |
| `dentro1km` | flag: parcel within 1 km of something (a reference layer/boundary) |
| `COD_INEI`, `N_SEA` | INEI census linkage fields — empty in this export |

In [20]:
# Key quality checks on the polygon layer
print("COD_PREDIO — unique:", cad_attrs['COD_PREDIO'].nunique(),
      "| null:", cad_attrs['COD_PREDIO'].isna().sum())
print()
for c in ["FUENTE", "COND_JUR", "dentro1km"]:
    print(cad_attrs[c].value_counts(dropna=False).to_string())
    print()
print("AREA_S_HA (ha):")
print(cad_attrs['AREA_S_HA'].describe())

COD_PREDIO — unique: 179608 | null: 10476

FUENTE
1    179722
2      9485
5       891

COND_JUR
1    188131
2      1363
3       604

dentro1km
NaN    169091
1.0     21007

AREA_S_HA (ha):
count    190098.000000
mean          2.434070
std         199.230978
min           0.000414
25%           0.193835
50%           0.473576
75%           1.125782
max       76322.159888
Name: AREA_S_HA, dtype: float64


~190k polygons, of which ~180k carry a `COD_PREDIO` (about 10k have a null code — geometry
with no parcel identifier, so unlinkable to attributes). `AREA_S_HA` has a sane median
(~0.47 ha) matching the cadastral `area_ha` from section 2, confirming these are the same
parcels — with a handful of huge outliers (max ~76,000 ha) that are almost certainly
multi-part/erroneous geometries to filter later.

### The companion `qgis_stefany/PIURA.dta`

This is **not** the same file as the cadastral `.dta` from section 2. It is the CENAGRO
**attribute table** for the polygon layer: the same cadastral columns as the shapefile's DBF
(`COD_PREDIO`, `UBIGEO`, `AREA_S_HA`, `FUENTE`, ...) plus census questionnaire fields
`P001`/`P002`/`P003` (department/province/district codes) and `P009_01..03` (holder
surname/name parts) and `NPARC_PETT`. It is read with `pyreadstat` and `encoding="latin1"`
because the file stores accented names (Ñ, etc.) in Latin-1 rather than UTF-8, which trips the
default reader.

In [21]:
import pyreadstat

cad_tab, _ = pyreadstat.read_dta(RAW / "qgis_stefany" / "PIURA.dta", encoding="latin1")
print("shape:", cad_tab.shape)
print("columns:", list(cad_tab.columns))
print("COD_PREDIO — unique:", cad_tab['COD_PREDIO'].nunique(),
      "| null:", cad_tab['COD_PREDIO'].isna().sum())
cad_tab.head()

shape: (158638, 15)
columns: ['COD_INEI', 'UBIGEO', 'NUM_PREDIO', 'AREA_S_HA', 'N_SEA', 'COND_JUR', 'COD_PREDIO', 'FUENTE', 'P009_01', 'P009_02', 'P009_03', 'P001', 'P002', 'P003', 'NPARC_PETT']
COD_PREDIO — unique: 154540 | null: 0


,COD_INEI,UBIGEO,NUM_PREDIO,AREA_S_HA,N_SEA,COND_JUR,COD_PREDIO,FUENTE,P009_01,P009_02,P009_03,P001,P002,P003,NPARC_PETT
0,,200101,06074,5.745866,,1,7_5359445_06074,1,ABAD,CAMPOVERDE,MANUEL DE JESUS,20,01,01,1.0
1,,200101,16741,0.823470,,1,7_5359445_16741,1,ABAD,DELGADO,MARLENY,20,01,01,1.0
2,,200101,06694,1.175265,,1,7_5359450_06694,1,ABAD,GUTIERRES,MERCEDES,20,01,01,1.0
3,,200101,06684,7.769222,,1,7_5359450_06684,1,ABAD,GUTIERREZ,ISABEL,20,01,01,1.0
4,,200101,06667,8.440842,,1,7_5359450_06667,1,ABAD,GUTIERREZ,PEDRO,20,01,01,1.0


### Visual check — one district's parcels over satellite imagery

Plotting all 190k polygons at department scale would be an unreadable smear, so we load the
geometry for a single district (`UBIGEO 200402`, ~2.4k parcels) and overlay it on Esri
satellite imagery. If the shapefile is sound, the polygons should tile cleanly over visible
field boundaries.

In [22]:
DISTRICT = "200402"

cad_dist = gpd.read_file(SHP_PATH, where=f"UBIGEO = '{DISTRICT}'").to_crs(3857)
print(f"parcels in district {DISTRICT}:", len(cad_dist))

fig, ax = plt.subplots(figsize=(11, 11), dpi=200)
cad_dist.boundary.plot(ax=ax, color="yellow", linewidth=0.4, zorder=2)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zorder=0)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.DarkMatterOnlyLabels, zorder=1)
ax.set_title(f"CENAGRO parcel polygons over satellite imagery — district {DISTRICT} "
             f"(n={len(cad_dist):,})")
ax.set_axis_off()
plt.tight_layout()
plt.show()

parcels in district 200402: 2449


<Figure size 2200x2200 with 1 Axes>

## 6. Can we now link polygons → crops? (the whole point)

The polygons carry `COD_PREDIO`. The cadastral table from section 2 (`piura_dta`) carries
**both** `COD_PREDIO` and `CodigoSSET`. The SSET workbook (`piura`) carries `Codigo SSET` and
the crop label `CULTIVO`. So the join chain is:

```
polygon (COD_PREDIO)  ──►  piura_dta (COD_PREDIO → CodigoSSET)  ──►  piura (Codigo SSET → CULTIVO)
   geometry                    cadastral bridge                        crop label + date
```

We reuse the `piura` and `piura_dta` frames already loaded above, so no files are re-read.

In [23]:
# Sets for each link in the chain
poly_codpred = set(cad_attrs['COD_PREDIO'].dropna().astype(str))

bridge = piura_dta[['COD_PREDIO', 'CodigoSSET']].copy()
bridge['COD_PREDIO'] = bridge['COD_PREDIO'].astype(str)
bridge['CodigoSSET'] = bridge['CodigoSSET'].astype(str)

sset_with_crop = set(piura.loc[piura['CULTIVO'].notna(), 'Codigo SSET'].astype(str))

# Link 1: polygon -> cadastral bridge
bridge_in_poly = bridge[bridge['COD_PREDIO'].isin(poly_codpred)]
print(f"Cadastral parcels with a matching polygon: {bridge_in_poly['COD_PREDIO'].nunique():,} "
      f"of {bridge['COD_PREDIO'].nunique():,} "
      f"({bridge_in_poly['COD_PREDIO'].nunique()/bridge['COD_PREDIO'].nunique():.1%})")

# Link 2: bridge -> crop label
labelled = bridge_in_poly[bridge_in_poly['CodigoSSET'].isin(sset_with_crop)]
print(f"...of those, polygons reaching >=1 labelled CULTIVO record: "
      f"{labelled['COD_PREDIO'].nunique():,}")

Cadastral parcels with a matching polygon: 156,672 of 158,420 (98.9%)
...of those, polygons reaching >=1 labelled CULTIVO record: 66,363


### Verdict — the geometry gap is closed

The chain holds: **~99% of cadastral parcels now have a real boundary polygon**, and roughly
**58k parcels link all the way through to at least one crop label**. That is a workable
training set of *(polygon, crop, approximate date)* triples — exactly the shape the classifier
needs, and a large improvement over the centroid-only approximation proposed in section 4.

**What each file contributes to the final training table:**

| piece | source file | provides |
|---|---|---|
| parcel geometry | `qgis_stefany/…FINAL.shp` | polygon boundary (clip region for imagery) |
| `COD_PREDIO ↔ CodigoSSET` bridge | `grafica_tabular_catastro_Piura.dta` | joins geometry to crop records |
| crop label + date | `BD SSET(...).xlsx` | `CULTIVO`, `FECHA EMPADRONAMIENTO` (per parcel-year) |

**Provenance note:** the `.shp.xml` metadata records the source as `CENAGRO.gdb`, stamped
2012-04-10 — this is the geometry layer of Peru's **2012 agricultural census (CENAGRO)** for
Piura. That pins the polygons to a *2012* land state, which matters for imagery: parcel
boundaries drawn in 2012 are most trustworthy for scenes from around that period, and crop
records stamped with far-off (or stub) dates should be treated with care when a boundary may
since have changed.

**Caveats that remain before modelling:**
- ~10k polygons have a null `COD_PREDIO` and can't be linked — drop them.
- Only parcels whose SSET record has a non-null `CULTIVO` are usable as labels (~58k); the rest
  are geometry without a target.
- A parcel is a *panel* (section 1): one polygon can map to several crop-years. Keep one row per
  parcel-year and split train/test **by parcel** to avoid leakage.
- The date-reliability problem from section 4 is unchanged — polygons fix *where*, not *when*.

**Updated next step:** build the joined GeoDataFrame — dissolve the shapefile to one polygon
per `COD_PREDIO`, attach `CodigoSSET` via the cadastral bridge, join the SSET crop-years, drop
fallow/blank `CULTIVO`, and hand the resulting `(geometry, CULTIVO, year)` table to the GEE
extraction step.

## 7. Joining the datasets and a visual ground-truth check

This section turns the three sources into a single **labelled polygon layer** and then plots a
small area, colouring each parcel boundary by its declared crop, over satellite imagery. The
purpose is a sanity check: do the CENAGRO polygons sit on real fields, and does the declared
`CULTIVO` look consistent with what the imagery shows?

We reuse `piura` (SSET crop records) and `piura_dta` (the `COD_PREDIO ↔ CodigoSSET` bridge)
already in memory, and read the polygon geometry from the shapefile.

In [24]:
# --- 7a. One crop label per parcel, attached to its polygon ---
# Labels that are not an identifiable growing crop (fallow, "not specified", etc.)
FALLOW = {"EN DESCANSO", "DESCANSO", "TERRENO EN DESCANSO", "TERRENO EN DESCANZO",
          "TERRENO GRADEADO", "NO ESPECIFICA", "RIEGO"}

crop_src = piura.loc[piura["CULTIVO"].notna(),
                     ["Codigo SSET", "CULTIVO", "FECHA EMPADRONAMIENTO"]].copy()
crop_src["Codigo SSET"] = crop_src["Codigo SSET"].astype(str)
crop_src["CULTIVO"] = crop_src["CULTIVO"].replace({"ALLGODON": "ALGODON"})  # fix an obvious typo
crop_src = crop_src[~crop_src["CULTIVO"].isin(FALLOW)]

# A parcel is a panel (section 1); keep its most recent declaration as the representative label.
crop_1 = (crop_src.sort_values("FECHA EMPADRONAMIENTO")
                  .groupby("Codigo SSET", as_index=False).last()
                  .rename(columns={"Codigo SSET": "CodigoSSET"}))

# bridge: COD_PREDIO <-> CodigoSSET, from the cadastral table (section 2)
bridge = piura_dta[["COD_PREDIO", "CodigoSSET"]].copy()
bridge["COD_PREDIO"] = bridge["COD_PREDIO"].astype(str)
bridge["CodigoSSET"] = bridge["CodigoSSET"].astype(str)

label_by_pred = (bridge.merge(crop_1, on="CodigoSSET", how="inner")
                       .drop_duplicates("COD_PREDIO")
                       [["COD_PREDIO", "CULTIVO", "FECHA EMPADRONAMIENTO"]])

# attach labels to polygon geometry
parcels_geom = gpd.read_file(SHP_PATH, columns=["COD_PREDIO"])
parcels_geom["COD_PREDIO"] = parcels_geom["COD_PREDIO"].astype(str)
parcels_labelled = (parcels_geom.merge(label_by_pred, on="COD_PREDIO", how="inner")
                                .to_crs(4326))

print(f"labelled polygons: {len(parcels_labelled):,}")
parcels_labelled["CULTIVO"].value_counts().head(12)

labelled polygons: 56,787


CULTIVO
ARROZ             25383
MAIZ               6561
ALGODON            3485
TRIGO              1197
PASTO ELEFANTE      974
MANGO               832
PASTO               666
LIMON               618
FREJOL              557
PLATANO             553
CAFE                403
CAÑA DE AZUCAR      374
Name: count, dtype: int64[pyarrow]

### Choosing an area to inspect

We want a compact window holding several parcels and at least two different crops. The box
below sits in an irrigated valley (district 200801) where cotton (`ALGODON`), rice (`ARROZ`)
and maize (`MAIZ`) parcels are interleaved — three crops with visually distinct signatures
(flooded/bunded rice paddies vs. row-cropped cotton and maize), which makes the check
meaningful.

In [25]:
# --- 7b. Compact area of interest (lon/lat box), restricted to the three focus crops ---
AOI = dict(minlon=-80.80945, maxlon=-80.80375, minlat=-5.52630, maxlat=-5.52127)
FOCUS = ["ARROZ", "ALGODON", "MAIZ"]

aoi = parcels_labelled.cx[AOI["minlon"]:AOI["maxlon"], AOI["minlat"]:AOI["maxlat"]].copy()
aoi = aoi[aoi["CULTIVO"].isin(FOCUS)]
print(f"{len(aoi)} parcels in view | crops: {aoi['CULTIVO'].value_counts().to_dict()}")
aoi[["COD_PREDIO", "CULTIVO"]].sort_values("CULTIVO").reset_index(drop=True)

22 parcels in view | crops: {'ALGODON': 11, 'ARROZ': 9, 'MAIZ': 2}


,COD_PREDIO,CULTIVO
0,7_5209385_60525,ALGODON
1,7_5209385_57390,ALGODON
2,7_5209385_60527,ALGODON
3,7_5209390_55495,ALGODON
4,7_5209390_55503,ALGODON
5,7_5209390_55507,ALGODON
6,7_5209390_55508,ALGODON
7,7_5209390_55517,ALGODON
8,7_5209385_60526,ALGODON
9,7_5209390_55515,ALGODON


In [26]:
# --- 7c. Plot: parcel boundaries coloured by declared crop, over satellite imagery ---
import matplotlib.patches as mpatches

CROP_COLORS = {"ARROZ": "#00e5ff", "ALGODON": "#ffe600", "MAIZ": "#ff2079"}  # cyan / yellow / magenta

aoi_web = aoi.to_crs(3857)
fig, ax = plt.subplots(figsize=(12, 12), dpi=200)
for crop, color in CROP_COLORS.items():
    part = aoi_web[aoi_web["CULTIVO"] == crop]
    if len(part):
        part.plot(ax=ax, facecolor=color, alpha=0.18, zorder=2)
        part.boundary.plot(ax=ax, color=color, linewidth=2.4, zorder=3)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom=17, zorder=0)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.DarkMatterOnlyLabels, zoom=17, zorder=1)

handles = [mpatches.Patch(facecolor=c, edgecolor="white",
                          label=f"{k.title()} (n={int((aoi['CULTIVO'] == k).sum())})")
           for k, c in CROP_COLORS.items()]
ax.legend(handles=handles, loc="upper right", framealpha=0.9, title="Declared crop (CULTIVO)")
ax.set_title("CENAGRO parcels coloured by declared crop, over satellite imagery\n"
             "visual ground-truth check — do polygons land on fields, and do crops agree?")
ax.set_axis_off()
plt.tight_layout()
plt.show()

<Figure size 2400x2400 with 1 Axes>

### How to read this, and the caveats

Each coloured outline is one CENAGRO parcel; its colour is the crop declared for that parcel in
the SSET registry. Two things to judge by eye:

1. **Location** — do the polygon boundaries follow the field edges, canals and access tracks
   visible in the imagery? If they do, the geometry↔attribute join is spatially sound.
2. **Crop agreement** — does each colour match the surface texture? Rice (`ARROZ`) parcels tend
   to look like flat, bunded, often wet/uniformly green paddies; cotton (`ALGODON`) and maize
   (`MAIZ`) look like drier row crops. Perfect agreement isn't expected (see below), but gross
   disagreement would flag a problem.

**Caveats that limit how literally to take this check:**
- **Time mismatch.** The `CULTIVO` label is the *most recent SSET declaration* for the parcel,
  and its date is unreliable (section 1). The Esri basemap is *recent* imagery, not the 2012
  CENAGRO epoch nor the declaration date — so the crop on the ground today can legitimately
  differ from the label. This is a plausibility spot-check, not validation.
- **One label per parcel.** Parcels are panels; collapsing to the latest record hides rotation.
- **Label noise.** Free-text `CULTIVO` needs cleaning (e.g. the `ALLGODON` typo fixed here);
  a production pipeline needs a proper controlled vocabulary.
- **Not a labelled-imagery pair yet.** For actual training we would pull imagery from a window
  matched to each parcel's crop season, clipped to its polygon — this plot just confirms the
  spatial join is trustworthy enough to build that on.

## 8. Multiple locations, with year-matched (Landsat) basemaps

Section 7 used a single area over a *recent* Esri mosaic. Here we do the same overlay —
parcel boundaries coloured **and labelled** by crop — but across **four locations**, and each
panel's basemap is **Landsat imagery from the same year the crop was recorded**.

**Why Landsat and not Sentinel-2.** The polygons come from the 2012 CENAGRO cadastre, and they
only join to SSET crop records from the *older* PETT registration waves — the linked crop years
run ~1998–2007 (see the year histogram in section 1/6), with essentially nothing after. That
predates Sentinel-2 (2015→). Landsat 5 TM (1984–2012) is the only sensor that actually covers
these years, so it is what makes "same year as the crop entry" possible. The cost is
resolution: at **30 m**, a ~0.5 ha parcel is only a handful of pixels, so the imagery looks
blocky compared with section 7 — that blockiness is a real property of the only year-matched
source available, not a plotting artefact.

Each location below was chosen to satisfy the brief — **≥3 parcels and ≥2 distinct crops** —
*and* to have a single dominant crop-entry year, so the basemap year is unambiguous.

In [27]:
# --- 8a. Earth Engine init + a year-matched Landsat-5 true-colour basemap helper ---
import ee, io, urllib.request
import matplotlib.image as mpimg

# SET THIS to your Google Cloud project id that has the Earth Engine API enabled,
# after running `earthengine authenticate` once in a terminal.
GEE_PROJECT = "peru-crop-classifier"

ee.Initialize(project=GEE_PROJECT)

def _mask_l5(img):
    # Collection-2 QA_PIXEL (same bits for L5 & L7): bit 3 = cloud, bit 4 = cloud shadow
    qa = img.select("QA_PIXEL")
    clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return img.updateMask(clear)

def landsat_rgb(bbox, year, dim=700):
    """Return (rgb_array, extent) for a cloud-masked Landsat-5 median true-colour
    composite over bbox=[minlon,minlat,maxlon,maxlat] for the given calendar year."""
    region = ee.Geometry.Rectangle(bbox)
    def _prep(coll_id):
        return (ee.ImageCollection(coll_id)
                  .filterBounds(region)
                  .filterDate(f"{year}-01-01", f"{year}-12-31")
                  .map(_mask_l5))
    # Merge Landsat-5 and Landsat-7 (identical SR_B* band naming in Collection-2);
    # L5 has spatial/temporal gaps in some year-locations (e.g. 2006 here), L7 fills them.
    col = _prep("LANDSAT/LT05/C02/T1_L2").merge(_prep("LANDSAT/LE07/C02/T1_L2"))
    # SR_B3/B2/B1 = red/green/blue; apply C2 scale to surface reflectance
    rgb = (col.median().select(["SR_B3", "SR_B2", "SR_B1"])
              .multiply(0.0000275).add(-0.2))
    url = rgb.getThumbURL({"region": region, "dimensions": dim, "format": "png",
                            "min": 0.0, "max": 0.3, "crs": "EPSG:4326"})
    arr = mpimg.imread(io.BytesIO(urllib.request.urlopen(url, timeout=90).read()))
    extent = [bbox[0], bbox[2], bbox[1], bbox[3]]  # [minlon,maxlon,minlat,maxlat]
    return arr, extent

print("Earth Engine ready; project:", GEE_PROJECT)

Earth Engine ready; project: peru-crop-classifier


In [28]:
# --- 8b. Clean, single-crop labels + the four locations ---
KEEP = {"ARROZ", "MAIZ", "ALGODON", "LIMON", "MANGO", "PLATANO", "CAFE",
        "YUCA", "TRIGO", "FREJOL", "CACAO", "CAMOTE", "ALGARROBO"}

crop_src = piura.loc[piura["CULTIVO"].notna(),
                     ["Codigo SSET", "CULTIVO", "FECHA EMPADRONAMIENTO"]].copy()
crop_src["Codigo SSET"] = crop_src["Codigo SSET"].astype(str)
crop_src["CULTIVO"] = crop_src["CULTIVO"].replace({"ALLGODON": "ALGODON"})
crop_src = crop_src[crop_src["CULTIVO"].isin(KEEP)]
crop_1 = (crop_src.sort_values("FECHA EMPADRONAMIENTO")
                  .groupby("Codigo SSET", as_index=False).last()
                  .rename(columns={"Codigo SSET": "CodigoSSET"}))
crop_1["YEAR"] = crop_1["FECHA EMPADRONAMIENTO"].dt.year

bridge = piura_dta[["COD_PREDIO", "CodigoSSET"]].copy()
bridge["COD_PREDIO"] = bridge["COD_PREDIO"].astype(str)
bridge["CodigoSSET"] = bridge["CodigoSSET"].astype(str)
label_by_pred = (bridge.merge(crop_1, on="CodigoSSET", how="inner")
                       .drop_duplicates("COD_PREDIO")[["COD_PREDIO", "CULTIVO", "YEAR"]])

parcels_clean = (parcels_geom.merge(label_by_pred, on="COD_PREDIO", how="inner").to_crs(4326))

# Four pre-scouted locations: (centre lon, lat), half-width in degrees, dominant crop-entry year
LOCATIONS = [
    {"name": "A", "lon": -80.05687, "lat": -5.17931, "hw": 0.0032, "year": 2001},
    {"name": "B", "lon": -80.39222, "lat": -4.89281, "hw": 0.0032, "year": 1999},
    {"name": "C", "lon": -80.00322, "lat": -4.63620, "hw": 0.0032, "year": 2006},
    {"name": "D", "lon": -80.23562, "lat": -5.00453, "hw": 0.0032, "year": 1999},
]
for loc in LOCATIONS:
    sub = parcels_clean.cx[loc["lon"]-loc["hw"]:loc["lon"]+loc["hw"],
                           loc["lat"]-loc["hw"]:loc["lat"]+loc["hw"]]
    sub = sub[sub["YEAR"] == loc["year"]]
    print(f"Location {loc['name']} ({loc['year']}): {len(sub)} parcels, "
          f"crops={sub['CULTIVO'].value_counts().to_dict()}")

Location A (2001): 15 parcels, crops={'MANGO': 4, 'ALGODON': 3, 'MAIZ': 3, 'LIMON': 3, 'PLATANO': 1, 'ALGARROBO': 1}
Location B (1999): 9 parcels, crops={'LIMON': 4, 'MANGO': 2, 'ALGARROBO': 2, 'MAIZ': 1}
Location C (2006): 10 parcels, crops={'ARROZ': 6, 'MAIZ': 2, 'YUCA': 1, 'MANGO': 1}
Location D (1999): 15 parcels, crops={'ALGODON': 5, 'MAIZ': 5, 'MANGO': 3, 'FREJOL': 2}


In [29]:
# --- 8c. 2x2 grid: parcel polygons coloured + labelled by crop, over same-year Landsat ---
# Global colour per crop so colours are consistent across panels.
all_crops = sorted(parcels_clean["CULTIVO"].unique())
palette = ["#00e5ff", "#ffe600", "#ff2079", "#7CFC00", "#ff8c00", "#c77dff",
           "#00ff9f", "#ff5555", "#4dabf7", "#ffd6a5", "#e0e0e0", "#b5179e", "#f4a261"]
CROP_COLORS = {c: palette[i % len(palette)] for i, c in enumerate(all_crops)}

fig, axes = plt.subplots(2, 2, figsize=(17, 17), dpi=170)
for ax, loc in zip(axes.ravel(), LOCATIONS):
    bbox = [loc["lon"]-loc["hw"], loc["lat"]-loc["hw"],
            loc["lon"]+loc["hw"], loc["lat"]+loc["hw"]]
    sub = parcels_clean.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    sub = sub[sub["YEAR"] == loc["year"]].copy()

    img, extent = landsat_rgb(bbox, loc["year"])
    ax.imshow(img, extent=extent, origin="upper", zorder=0)
    ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])

    for crop, part in sub.groupby("CULTIVO"):
        part.boundary.plot(ax=ax, color=CROP_COLORS[crop], linewidth=2.0, zorder=2)
    # crop-name label at each parcel centroid
    for _, row in sub.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["CULTIVO"], (c.x, c.y), color="white", fontsize=6.5,
                    ha="center", va="center", zorder=3,
                    path_effects=[__import__("matplotlib").patheffects.withStroke(
                        linewidth=1.6, foreground="black")])
    present = sub["CULTIVO"].value_counts()
    handles = [__import__("matplotlib").patches.Patch(color=CROP_COLORS[c], label=f"{c} ({n})")
               for c, n in present.items()]
    ax.legend(handles=handles, loc="upper right", fontsize=7, framealpha=0.85)
    ax.set_title(f"Location {loc['name']} — Landsat-5/7 {loc['year']}  "
                 f"({len(sub)} parcels, {present.size} crops)", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("CENAGRO parcels over year-matched Landsat-5/7 imagery (30 m) — "
             "boundary colour & label = declared crop", fontsize=13, y=0.995)
plt.tight_layout()
plt.show()

<Figure size 2890x2890 with 4 Axes>

### 8d. The same four locations at 10 m — earliest available Sentinel-2

The Landsat panels above are year-matched but coarse. Here we re-plot the **same four AOIs and
the same parcel polygons** on **Sentinel-2** imagery (10 m, ~9× finer than Landsat's 30 m) so
the boundaries can be checked against clearly-resolved field edges.

**Important — this is a *location* check, not a *crop* check.** Sentinel-2 only exists from mid
2015 onward, whereas these parcels' crop declarations are from 1999–2006. So each panel uses the
*earliest low-cloud Sentinel-2 scene available* (late 2015 / 2016, shown in each title), which is
~10–17 years **after** the crop was recorded. The crop labels are kept for identification, but
the surface below them is a different era — judge whether the polygons fall on real fields and
follow real boundaries, **not** whether the crop matches.

In [30]:
# --- 8d. Same AOIs/polygons on the earliest available Sentinel-2 (10 m) ---
def sentinel2_rgb(bbox, dim=700, max_cloud=20, start="2015-06-23", end="2018-12-31"):
    """Earliest low-cloud Sentinel-2 (TOA, harmonized) true-colour scene over bbox.
    Returns (rgb_array, extent, date_str)."""
    region = ee.Geometry.Rectangle(bbox)
    col = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
             .filterBounds(region)
             .filterDate(start, end)
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
             .sort("system:time_start"))
    img = col.first()
    date = img.date().format("YYYY-MM-dd").getInfo()
    rgb = img.select(["B4", "B3", "B2"]).divide(10000)   # 10 m R/G/B, TOA -> reflectance
    url = rgb.getThumbURL({"region": region, "dimensions": dim, "format": "png",
                            "min": 0.0, "max": 0.3, "crs": "EPSG:4326"})
    arr = mpimg.imread(io.BytesIO(urllib.request.urlopen(url, timeout=90).read()))
    return arr, [bbox[0], bbox[2], bbox[1], bbox[3]], date

fig, axes = plt.subplots(2, 2, figsize=(17, 17), dpi=170)
for ax, loc in zip(axes.ravel(), LOCATIONS):
    bbox = [loc["lon"]-loc["hw"], loc["lat"]-loc["hw"],
            loc["lon"]+loc["hw"], loc["lat"]+loc["hw"]]
    sub = parcels_clean.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    sub = sub[sub["YEAR"] == loc["year"]].copy()

    img, extent, s2date = sentinel2_rgb(bbox)
    ax.imshow(img, extent=extent, origin="upper", zorder=0)
    ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])

    for crop, part in sub.groupby("CULTIVO"):
        part.boundary.plot(ax=ax, color=CROP_COLORS[crop], linewidth=2.0, zorder=2)
    for _, row in sub.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["CULTIVO"], (c.x, c.y), color="white", fontsize=6.5,
                    ha="center", va="center", zorder=3,
                    path_effects=[__import__("matplotlib").patheffects.withStroke(
                        linewidth=1.6, foreground="black")])
    present = sub["CULTIVO"].value_counts()
    handles = [__import__("matplotlib").patches.Patch(color=CROP_COLORS[c], label=f"{c} ({n})")
               for c, n in present.items()]
    ax.legend(handles=handles, loc="upper right", fontsize=7, framealpha=0.85)
    ax.set_title(f"Location {loc['name']} — Sentinel-2 {s2date} (10 m)  "
                 f"[crop declared {loc['year']}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Same parcels on earliest Sentinel-2 (10 m) — high-res LOCATION check "
             "(imagery is 2015-16, crop labels are 1999-2006)", fontsize=13, y=0.995)
plt.tight_layout()
plt.show()

<Figure size 2890x2890 with 4 Axes>

At 10 m the parcel outlines can be compared against genuinely resolved field boundaries,
canals and tracks — a much stronger test of the *geometry* than the blocky Landsat panels.
Where an outline hugs a visible field edge, the polygon↔attribute join is spatially trustworthy.
Keep in mind the decade-plus gap: any mismatch between a label and the surface is expected here
(different year, and fields get replanted/subdivided), so this panel speaks only to *where* the
parcels are, while section 8c speaks to *when*. Together they bracket the trade-off — Sentinel-2
gives resolution, Landsat gives the matching year, and no single source gives both for this data.

### 8e. The same four locations on sub-metre Esri imagery (recent)

Finally, the same AOIs and polygons over **Esri World Imagery** — a recent, sub-metre mosaic
(no year selection, like section 7). This is the sharpest available view, so it is the strongest
possible *location* check: at this resolution individual trees, furrows and field edges are
visible, and a well-registered polygon should trace them closely.

Same caveat as 8d, only more so: this imagery is *current*, ~15-25 years after the crop
declarations, so it speaks purely to *where* the parcels are, not what grew on them. Reading the
three basemaps together: **8c** = coarse but year-matched (30 m Landsat), **8d** = 10 m from
2015-16, **8e** = sub-metre but present-day.

In [31]:
# --- 8e. Same AOIs/polygons on recent sub-metre Esri World Imagery ---
from shapely.geometry import box

fig, axes = plt.subplots(2, 2, figsize=(17, 17), dpi=170)
for ax, loc in zip(axes.ravel(), LOCATIONS):
    bbox = [loc["lon"]-loc["hw"], loc["lat"]-loc["hw"],
            loc["lon"]+loc["hw"], loc["lat"]+loc["hw"]]
    sub = parcels_clean.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    sub = sub[sub["YEAR"] == loc["year"]].to_crs(3857)

    # lock axes to the AOI extent (in Web Mercator) so contextily fetches the right tiles
    b = gpd.GeoSeries([box(*bbox)], crs=4326).to_crs(3857).total_bounds
    ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3])

    for crop, part in sub.groupby("CULTIVO"):
        part.boundary.plot(ax=ax, color=CROP_COLORS[crop], linewidth=2.0, zorder=2)
    for _, row in sub.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["CULTIVO"], (c.x, c.y), color="white", fontsize=6.5,
                    ha="center", va="center", zorder=3,
                    path_effects=[__import__("matplotlib").patheffects.withStroke(
                        linewidth=1.6, foreground="black")])
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom=17, zorder=0)

    present = sub["CULTIVO"].value_counts()
    handles = [__import__("matplotlib").patches.Patch(color=CROP_COLORS[c], label=f"{c} ({n})")
               for c, n in present.items()]
    ax.legend(handles=handles, loc="upper right", fontsize=7, framealpha=0.85)
    ax.set_title(f"Location {loc['name']} — Esri World Imagery (recent, sub-metre)  "
                 f"[crop declared {loc['year']}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Same parcels on recent sub-metre Esri imagery — sharpest LOCATION check "
             "(imagery is present-day, crop labels are 1999-2006)", fontsize=13, y=0.995)
plt.tight_layout()
plt.show()

<Figure size 2890x2890 with 4 Axes>

### How to read this, and the caveats

Four locations, each a self-contained check like section 7 but with the basemap pulled from
Landsat-5/7 for **the year that location's crop was recorded** (2001, 1999, 2006, 1999). Every
polygon is outlined and captioned with its declared crop.

- **Resolution.** 30 m Landsat (5 & 7) over ~0.5 ha parcels means only a few pixels per field — the
  imagery is deliberately blocky. Boundaries and their agreement with tonal blocks are still
  legible, but fine field texture is not resolvable the way it was in section 7's sub-metre
  imagery. This is the price of temporal fidelity with the only sensor that covers these years.
- **Now the year *does* match.** Unlike section 7, the picture underneath is contemporaneous
  with the crop declaration, so a rice/cotton/mango label and the surface it sits on should
  correspond far better — modulo the reliability of `FECHA EMPADRONAMIENTO` itself (still a
  caveat from section 1; here we at least required a single dominant year per location).
- **Cloud/compositing.** Each basemap is a cloud-masked median over that whole year. Piura's
  arid coast keeps this clean, but a median blends any within-year crop changes.
- **Label scope.** Restricted to clean single-crop names (`KEEP` set) for legibility; the messy
  free-text and intercropped entries (`"MANGO 70% LIMON 30%"`) are excluded here, not solved.

**Implication for training data.** This confirms the intended recipe end-to-end: parcel polygon
+ declared crop + a year → a matching satellite scene. For a real classifier the 30 m Landsat
signal is usable for large/among-field discrimination but marginal for the smallest parcels;
where crop years reach 2015+ (few, per section 6) Sentinel-2's 10 m would be far better. The
practical lever is the date problem, not the imagery: better per-parcel dates would sharpen both
the scene selection and the label.

## 9. Multiple crops per parcel, and different crops on neighbouring parcels

Looking at the section-8 panels, adjacent parcels with *different* declared crops looked
suspicious — in Piura you'd expect a farmer's neighbours to be growing the same thing. This
section quantifies that intuition in three parts:

1. **Multiple crops on one parcel** — over time (successive declarations) and within a single
   declaration (intercropping labels like `CAFE Y PLATANO`).
2. **Different crops on neighbouring parcels** — how often does it actually happen, and is it
   more or less than you'd expect by chance?
3. A wide-area map showing the real spatial pattern.

**A caveat that reframes the section-8 impression up front:** the four AOIs in section 8 were
*deliberately selected* to contain ≥2 crops (that was the brief). So they over-represent the
rare mixed case — they are **not** a random sample of Piura, and the "odd" adjacencies you saw
are exactly the exceptions this section is about counting.

In [32]:
# --- 9a. Multiple crops on the SAME parcel over time ---
KEEP = {"ARROZ", "MAIZ", "ALGODON", "LIMON", "MANGO", "PLATANO", "CAFE",
        "YUCA", "TRIGO", "FREJOL", "CACAO", "CAMOTE", "ALGARROBO"}

recs = piura.dropna(subset=["CULTIVO"]).copy()
per = recs.groupby("Codigo SSET")["CULTIVO"]
n_records = per.size()

# distinct *clean* crops per parcel (whitelist only, so typos/free-text don't inflate it)
clean = recs[recs["CULTIVO"].isin(KEEP)]
n_distinct = clean.groupby("Codigo SSET")["CULTIVO"].nunique()

print(f"parcels with a crop record:            {n_records.size:,}")
print(f"  ...with >=2 records (any):           {(n_records>=2).sum():,} "
      f"({(n_records>=2).mean():.1%})")
print(f"  ...that grew >=2 DIFFERENT clean crops over time: {(n_distinct>=2).sum():,} "
      f"({(n_distinct>=2).mean():.1%} of parcels with a clean crop)")

ax = (n_distinct.clip(upper=3).value_counts().sort_index()
      .rename({1:"1 crop",2:"2 crops",3:"3+ crops"})
      .plot(kind="bar", figsize=(6,3), title="Distinct crops declared per parcel over time (clean labels)"))
ax.set_ylabel("parcels"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

parcels with a crop record:            98,348
  ...with >=2 records (any):           67,512 (68.6%)
  ...that grew >=2 DIFFERENT clean crops over time: 589 (1.1% of parcels with a clean crop)


<Figure size 600x300 with 1 Axes>

So repeat visits are common (most parcels have ≥2 records), but the crop **label stays the
same** the vast majority of the time — only a small share of parcels ever record a genuinely
different crop across years. Multi-record ≠ multi-crop; the panel is mostly the same crop
re-declared.

In [33]:
# --- 9b. Multiple crops within a SINGLE declaration (intercropping / mixed labels) ---
# denominator = actual crop declarations (rows that HAVE a crop), not the null-crop rows
crop_recs = piura.dropna(subset=["CULTIVO"])
multi = crop_recs["CULTIVO"].str.contains(r"(?:\bY\b|,|/|\+|%)", regex=True, na=False)
print(f"declarations that list more than one crop: {multi.sum():,} "
      f"({multi.mean():.1%} of crop declarations)")
print("\nMost common mixed-crop labels:")
top_mixed = crop_recs.loc[multi, "CULTIVO"].value_counts().head(10)
print(top_mixed.to_string())

ax = top_mixed.iloc[::-1].plot(kind="barh", figsize=(8,4),
        title="Top intercrop / mixed-crop declarations")
ax.set_xlabel("records"); plt.tight_layout(); plt.show()

declarations that list more than one crop: 25,231 (14.2% of crop declarations)

Most common mixed-crop labels:
CULTIVO
CAFE Y PLATANO                             820
PLATANO Y CAFE                             568
ARROZ Y MAIZ                               341
LIMON Y MANGO                              238
NIVELACION Y GRADEO                        232
MAIZ Y FREJOL                              229
CAFE 50%-PLATANO 50%                       227
MANGO, LIMON                               210
EN DESCANSO, EXISTEN RASTROJOS DE ARROZ    207
MANGO Y LIMON                              194


<Figure size 800x400 with 1 Axes>

Intercropping is real and common in Piura (~1 in 7 declarations): coffee+banana in the
sierra, mango+lime and rice+maize on the coast. For a single-label classifier these are a
genuine complication — a mixed parcel isn't one class. They were excluded from the section-8
plots (whitelist of clean single crops), not solved.

In [34]:
# --- 9c. Do NEIGHBOURING parcels share a crop? ---
# Use the same clean, one-crop-per-parcel layer as section 8 (parcels_clean).
gp = parcels_clean.to_crs(32717)
gp = gp[gp.geometry.notna() & gp.geometry.is_valid].reset_index(drop=True)

# adjacency: buffer 5 m and self-join on intersects (robust to cadastral edge slivers)
gb = gp.copy(); gb["geometry"] = gb.geometry.buffer(5)
sj = gpd.sjoin(gb[["CULTIVO", "geometry"]],
               gp[["CULTIVO", "geometry"]].rename(columns={"CULTIVO": "CULTIVO_r"}),
               predicate="intersects")
sj = sj[sj.index != sj["index_right"]]
a = np.minimum(sj.index.values, sj["index_right"].values)
b = np.maximum(sj.index.values, sj["index_right"].values)
pairs = pd.DataFrame({"a": a, "b": b,
                      "c1": sj["CULTIVO"].values, "c2": sj["CULTIVO_r"].values}).drop_duplicates(["a", "b"])
same = (pairs["c1"] == pairs["c2"])

# null baseline: reshuffle crop labels across parcels, recompute neighbour-match rate
rng = np.random.default_rng(0)
crops = gp["CULTIVO"].values
null_rates = []
for _ in range(20):
    cmap = dict(enumerate(rng.permutation(crops)))
    null_rates.append((pairs["a"].map(cmap) == pairs["b"].map(cmap)).mean())

print(f"neighbouring parcel pairs analysed: {len(pairs):,}")
print(f"  SAME crop:      {same.mean():.1%}")
print(f"  DIFFERENT crop: {(~same).mean():.1%}")
print(f"  expected same-crop rate if crops were placed at random: "
      f"{np.mean(null_rates):.1%} (+/- {np.std(null_rates):.1%})")

diff = pairs[~same].copy()
diff["combo"] = [" + ".join(sorted(p)) for p in zip(diff["c1"], diff["c2"])]
top_combo = diff["combo"].value_counts().head(10)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(["same crop", "different"], [same.mean()*100, (~same).mean()*100],
          color=["#2a9d8f", "#e76f51"])
ax[0].axhline(np.mean(null_rates)*100, ls="--", color="gray")
ax[0].annotate("random-placement\nsame-crop rate", (0.5, np.mean(null_rates)*100+2),
               ha="center", fontsize=8, color="gray")
ax[0].set_ylabel("% of neighbour pairs"); ax[0].set_title("Neighbouring parcels: same vs different crop")
top_combo.iloc[::-1].plot(kind="barh", ax=ax[1], color="#e76f51")
ax[1].set_xlabel("neighbour pairs"); ax[1].set_title("Which crops end up adjacent (the exceptions)")
plt.tight_layout(); plt.show()

neighbouring parcel pairs analysed: 43,707
  SAME crop:      85.8%
  DIFFERENT crop: 14.2%
  expected same-crop rate if crops were placed at random: 43.5% (+/- 0.3%)


<Figure size 1300x400 with 2 Axes>

**This is the answer to the intuition.** ~86% of neighbouring parcels share the same crop
— and against a null where crops are scattered at random you'd only expect ~46%. So parcels are
strongly **clustered into single-crop blocks**, exactly as you suspected: a farmer's neighbours
are usually growing the same thing. Different-crop borders are the ~14% minority, and they are
dominated by combinations of *annual field crops that share the same irrigated systems and
rotate* — **rice+maize, cotton+rice, cotton+maize** make up two-thirds of them — which is
agronomically plausible (neighbours at different points in the same rotation), not obviously
wrong. Orchard-next-to-field mixes (mango/lime vs a field crop) are much rarer.

In [35]:
# --- 9d. The block pattern on a wider area (fill = declared crop) ---
CLON, CLAT, HW = -80.75, -5.39, 0.012
win = parcels_clean.cx[CLON-HW:CLON+HW, CLAT-HW:CLAT+HW].copy()
present = win["CULTIVO"].value_counts()
top6 = list(present.head(6).index)
palette = {"ARROZ":"#00b4d8","MAIZ":"#ffb703","ALGODON":"#e9ecef","MANGO":"#fb8500",
           "PLATANO":"#8ecae6","FREJOL":"#43aa8b","LIMON":"#f4d35e","YUCA":"#bc6c25"}
win["color"] = win["CULTIVO"].map(lambda c: palette.get(c, "#adb5bd"))

fig, ax = plt.subplots(figsize=(11, 11), dpi=170)
win.plot(ax=ax, color=win["color"], edgecolor="white", linewidth=0.15)
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=palette.get(c, "#adb5bd"), label=f"{c} ({present[c]})") for c in top6]
ax.legend(handles=handles, loc="upper right", title="Declared crop", framealpha=0.9)
ax.set_title(f"Declared crop by parcel over a ~2.5 km window ({len(win):,} parcels)\n"
             "same-crop parcels form contiguous blocks; mixed borders are the exception")
ax.set_axis_off(); plt.tight_layout(); plt.show()

<Figure size 1870x1870 with 1 Axes>

Zoomed out, the pattern is unmistakable: broad **contiguous blocks of a single crop**
(mostly rice here) with maize/cotton forming their own patches, and only thin seams where
different crops meet. That is why the hand-picked section-8 windows looked odd — they were
chosen to sit *on* those rare seams. Read together:

- **Multiple crops per parcel** are mostly an artefact of re-declaration (same crop) or genuine
  **intercropping labels** (~14% of records) — the latter is a real modelling complication.
- **Different crops on adjacent parcels** are real but uncommon (~14% of borders), spatially
  clustered, and dominated by rotating annual crops — consistent with your sense that Piura
  grows in uniform blocks.

**Implication for the classifier:** crop labels have strong spatial autocorrelation, so (i) a
train/test split must be blocked by area, not random parcels, or neighbours leak across the
split and inflate accuracy; and (ii) the mixed/intercrop labels need an explicit policy (drop,
multi-label, or a "mixed" class) rather than being forced into one crop.

## 10. Quick look-up: parcels around a coordinate

A small reusable helper to drop a point and see the CENAGRO parcels around it on high-res
Esri imagery. Parcels that carry a crop label are coloured and captioned; parcels with no
linked crop record are drawn as plain white outlines. Handy for spot-checking any lat/lon.

In [36]:
# --- 10. plot parcels near a coordinate ------------------------------------------------
# COD_PREDIO -> crop lookup (from the labelled layer built in section 7)
crop_lut = parcels_labelled[["COD_PREDIO", "CULTIVO"]].drop_duplicates("COD_PREDIO")
_PAL = ["#00e5ff", "#ffe600", "#ff2079", "#7CFC00", "#ff8c00", "#c77dff",
        "#00ff9f", "#ff5555", "#4dabf7", "#ffd6a5", "#b5179e", "#f4a261"]

def plot_point_parcels(lat, lon, radius_m=400, zoom=17, ax=None):
    """Mark (lat, lon) and show CENAGRO parcels within radius_m, over Esri imagery."""
    # find nearby parcels in the shapefile's native UTM CRS, then go to Web Mercator
    pt_utm = gpd.GeoSeries.from_xy([lon], [lat], crs=4326).to_crs(parcels_geom.crs)
    x, y = pt_utm.x[0], pt_utm.y[0]
    sub = parcels_geom.cx[x-radius_m:x+radius_m, y-radius_m:y+radius_m].merge(
        crop_lut, on="COD_PREDIO", how="left").to_crs(3857)
    labelled = sub["CULTIVO"].notna().sum()

    crops = sorted(sub["CULTIVO"].dropna().unique())
    cmap = {c: _PAL[i % len(_PAL)] for i, c in enumerate(crops)}

    # square window centred on the point, sized to the radius
    px, py = gpd.GeoSeries.from_xy([lon], [lat], crs=4326).to_crs(3857).iloc[0].coords[0]
    r = radius_m * 1.15
    if ax is None:
        _, ax = plt.subplots(figsize=(11, 11), dpi=170)
    ax.set_xlim(px-r, px+r); ax.set_ylim(py-r, py+r)

    # unlabelled parcels: white outline; labelled: coloured outline + caption
    un = sub[sub["CULTIVO"].isna()]
    if len(un):
        un.boundary.plot(ax=ax, color="white", linewidth=1.6, zorder=2)
    for crop, part in sub.dropna(subset=["CULTIVO"]).groupby("CULTIVO"):
        part.boundary.plot(ax=ax, color=cmap[crop], linewidth=2.4, zorder=3)
        for _, row in part.iterrows():
            c = row.geometry.centroid
            ax.annotate(crop, (c.x, c.y), color="white", fontsize=7, ha="center", va="center",
                        zorder=4, path_effects=[__import__("matplotlib").patheffects.withStroke(
                            linewidth=1.6, foreground="black")])
    ax.scatter([px], [py], marker="*", s=320, c="red", edgecolor="white",
               linewidth=1.2, zorder=5)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom=zoom, zorder=0)
    ax.set_title(f"Parcels within {radius_m} m of ({lat:.6f}, {lon:.6f})\n"
                 f"{len(sub)} parcels ({labelled} with a crop label)", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    return ax

plot_point_parcels(-4.964697, -80.679395)
plot_point_parcels(-4.987652, -80.684152)
plt.tight_layout(); plt.show()

<Figure size 1870x1870 with 1 Axes>

<Figure size 1870x1870 with 1 Axes>

## 11. Re-check: are the `qgis_stefany` polygons CENAGRO or PETT? + a better bridge

Two follow-ups:

1. **Provenance.** Earlier sections called the polygons "the 2012 CENAGRO cadastral layer" because
   the file is named `CATASTRO_CENAGRO_...` and its metadata points at a `CENAGRO.gdb`. But the word
   is that the *geometry* actually comes from **PETT** land-titling records, and only the files with
   "cenagro" in the name are census-derived. This section re-examines the metadata thoroughly to
   settle it.
2. **A wider bridge.** Section 8 concluded "the linked crop years run ~1998–2007" using
   `grafica_tabular_catastro_Piura.dta` as the polygon↔crop bridge. Here we redo that with the
   larger `grafica_tabular_Piura.dta` bridge to see whether it recovers more crops/polygons **and
   whether it extends the year range** (which would matter for Sentinel-2).

### 11.1 Provenance evidence #1 — the shapefile's own lineage (`.shp.xml`)

ESRI metadata records the geoprocessing history. We read it straight from the `.shp.xml`.

In [37]:
import re, pathlib

xml = pathlib.Path("../data/raw/qgis_stefany/CATASTRO_CENAGRO_PIURA_WGS84_Z17S_FINAL.shp.xml"
                   ).read_text(encoding="utf-8", errors="replace")

loc   = re.search(r"<linkage[^>]*>([^<]+)</linkage>", xml)
title = re.search(r"<resTitle[^>]*>([^<]+)</resTitle>", xml)
crea  = re.search(r"<CreaDate>(\d+)</CreaDate>", xml)
print("Feature-class title :", title.group(1) if title else "?")
print("Stored in           :", loc.group(1) if loc else "?")
print("GDB creation date   :", crea.group(1) if crea else "?")
print("\nLineage — how the layer was built (ArcGIS Process steps):")
for date, cmd in re.findall(r'<Process[^>]*Date="(\d+)"[^>]*>([^<]+)</Process>', xml):
    print(f"  {date}:  {cmd.strip()[:110]}")

Feature-class title : PIURA
Stored in           : file://\\PC33055\D$\00_CENAGRO\CENAGRO.gdb
GDB creation date   : 20120410

Lineage — how the layer was built (ArcGIS Process steps):
  20120229:  CopyFeatures D:\00_CENAGRO\PIURA.shp D:\00_CENAGRO\CENAGRO.gdb\FDS17\PIURA # 0 0 0
  20120229:  CalculateField CAT_PIURA AREA_S_HA [Shape_Area]/10000 VB #
  20120229:  CalculateField CAT_PIURA NOMBRE "SN" VB #
  20120229:  CalculateField CAT_PIURA COND_JUR 3 VB #
  20120229:  CalculateField CAT_PIURA FUENTE 5 VB #


Read the lineage carefully:

- The layer was **`CopyFeatures`-d from a pre-existing `PIURA.shp`** into `CENAGRO.gdb\FDS17\PIURA`
  — i.e. the geometry already existed as a standalone cadastral shapefile and was merely *imported*
  into a geodatabase that happens to be named `CENAGRO.gdb`.
- The feature class the fields are calculated on is **`CAT_PIURA`** — "**CAT**" = *catastro*
  (cadastre), the PETT/COFOPRI land-titling product, not a census layer.
- `CalculateField ... NOMBRE "SN"` and `... COND_JUR 3` / `FUENTE 5` are cadastral bookkeeping
  (owner name placeholder, juridical condition, source code) — titling concepts, not census ones.
- The `2012` dates are when this cadastre was *copied into the census geodatabase*, **not** when the
  parcels were surveyed (PETT titling long predates 2012).

### 11.2 Provenance evidence #2 — which fields are actually populated

If this were a census layer, the census identifiers (`COD_INEI`, `N_SEA` = *Sector de
Empadronamiento Agropecuario*) would be filled. If it is a PETT cadastre, the titling fields
(`COD_PREDIO`, `NUM_PREDIO`, `COND_JUR`) would be filled instead.

In [38]:
cad = gpd.read_file("../data/raw/qgis_stefany/CATASTRO_CENAGRO_PIURA_WGS84_Z17S_FINAL.shp",
                    ignore_geometry=True)
n = len(cad)
print(f"{n:,} features\n")
print(f"{'field':12s} {'non-null %':>10s}   role")
roles = {"COD_INEI":"CENSUS id (INEI)", "N_SEA":"CENSUS sector (SEA)",
         "COD_PREDIO":"PETT predio code", "NUM_PREDIO":"PETT predio number",
         "COND_JUR":"PETT juridical condition", "UBIGEO":"district code",
         "NOMBRE":"owner name", "FUENTE":"source code", "dentro1km":"derived flag (added later)"}
for f, role in roles.items():
    if f in cad.columns:
        print(f"{f:12s} {100*cad[f].notna().mean():9.1f}%   {role}")
print(f"\nNOMBRE == 'SN' (the placeholder from the lineage): {(cad['NOMBRE'].astype(str).str.strip()=='SN').sum():,}")

190,098 features

field        non-null %   role
COD_INEI           0.0%   CENSUS id (INEI)
N_SEA              0.0%   CENSUS sector (SEA)
COD_PREDIO        94.5%   PETT predio code
NUM_PREDIO        99.3%   PETT predio number
COND_JUR         100.0%   PETT juridical condition
UBIGEO           100.0%   district code
NOMBRE           100.0%   owner name
FUENTE           100.0%   source code
dentro1km         11.1%   derived flag (added later)

NOMBRE == 'SN' (the placeholder from the lineage): 27,593


**The census fields `COD_INEI` and `N_SEA` are 100% empty**; the PETT fields (`COD_PREDIO`,
`NUM_PREDIO`, `COND_JUR`) are populated. And 27,593 parcels carry `NOMBRE='SN'` — exactly the
placeholder written by the `CalculateField NOMBRE "SN"` lineage step. The attribute payload is
PETT, not census.

### 11.3 Provenance evidence #3 — the join key is a PETT/SSET key

`COD_PREDIO` is the PETT land-titling parcel code. If the geometry is PETT, it should key almost
perfectly into the PETT bridge tables.

In [39]:
poly = set(cad["COD_PREDIO"].dropna().astype(str))

def load_bridge(fname, enc=None):
    b, _ = pyreadstat.read_dta(f"../data/raw/{fname}", usecols=["COD_PREDIO", "CodigoSSET"], encoding=enc)
    b["COD_PREDIO"] = b["COD_PREDIO"].astype(str)
    b["CodigoSSET"] = b["CodigoSSET"].astype(str)
    return b.drop_duplicates()

bridge_cat  = load_bridge("grafica_tabular_catastro_Piura.dta")
bridge_full = load_bridge("grafica_tabular_Piura.dta", enc="latin1")

for name, b in [("catastro bridge", bridge_cat), ("full bridge (grafica_tabular_Piura)", bridge_full)]:
    codes = set(b["COD_PREDIO"])
    print(f"{name:38s}: {len(codes & poly):>7,} of its {len(codes):,} predios have a polygon "
          f"({100*len(codes & poly)/len(codes):.1f}%)")

catastro bridge                       : 108,695 of its 109,790 predios have a polygon (99.0%)
full bridge (grafica_tabular_Piura)   : 156,672 of its 158,420 predios have a polygon (98.9%)


### Provenance verdict

Every strand points the same way: the polygons are a **PETT / COFOPRI land-titling cadastre**
(`CAT_PIURA`, keyed by `COD_PREDIO`, titling attributes populated, census attributes empty), that
was **imported into the 2012 CENAGRO geodatabase to serve as its parcel base layer**. So:

- The **geometry is PETT** — consistent with what you were told ("everything except the
  census-named files is PETT"). The `CATASTRO_CENAGRO_...` name and the `CENAGRO.gdb` container
  refer to *where the cadastre was packaged* (the census used it as a reference frame), **not** to
  the origin of the parcels.
- Earlier sections' phrasing ("the 2012 CENAGRO cadastral layer") was therefore **imprecise** — it
  is a PETT cadastre bundled into the census GDB. The 2012 date is the packaging date, not a survey
  date. Nothing downstream breaks: `COD_PREDIO` still links to crops exactly as before; only the
  provenance label changes.

### 11.4 Re-run the "linked crop years" with the wider bridge

Now the second question: does swapping `grafica_tabular_catastro_Piura` → `grafica_tabular_Piura`
recover more crops/polygons, and does it push the year range past 2007?

In [40]:
# crop records reused from section 1 (piura); one row per record with a year
cr = piura.dropna(subset=["CULTIVO"])[["Codigo SSET", "FECHA EMPADRONAMIENTO"]].copy()
cr["CodigoSSET"] = cr["Codigo SSET"].astype(str)
cr["YEAR"] = cr["FECHA EMPADRONAMIENTO"].dt.year

def linked_years(bridge):
    bp = bridge[bridge["COD_PREDIO"].isin(poly)]                 # bridge rows whose polygon exists
    linked_sset = set(bp["CodigoSSET"])
    rec = cr[cr["CodigoSSET"].isin(linked_sset)]                 # crop records reachable to a polygon
    polys = bp[bp["CodigoSSET"].isin(set(cr["CodigoSSET"]))]["COD_PREDIO"].nunique()
    return rec, polys

rec_cat,  poly_cat  = linked_years(bridge_cat)
rec_full, poly_full = linked_years(bridge_full)

print(f"catastro bridge : {len(rec_cat):>7,} crop records reach a polygon | {poly_cat:,} polygons labelled")
print(f"full bridge     : {len(rec_full):>7,} crop records reach a polygon | {poly_full:,} polygons labelled")
print(f"gain            : +{len(rec_full)-len(rec_cat):,} records, +{poly_full-poly_cat:,} polygons")
print(f"records in 2015+: catastro={ (rec_cat['YEAR']>=2015).sum() }  full={ (rec_full['YEAR']>=2015).sum() }")

yc = pd.DataFrame({"catastro bridge": rec_cat["YEAR"].value_counts(),
                   "full bridge":     rec_full["YEAR"].value_counts()}).sort_index()
yc = yc[(yc.index >= 1996) & (yc.index <= 2019)]
ax = yc.plot(kind="bar", figsize=(12, 4), width=0.85,
             color=["#adb5bd", "#2a9d8f"])
ax.set_ylabel("linked crop records"); ax.set_xlabel("FECHA EMPADRONAMIENTO year")
ax.set_title("Crop records that reach a polygon, by year — catastro vs full bridge")
plt.tight_layout(); plt.show()

catastro bridge : 103,757 crop records reach a polygon | 57,971 polygons labelled
full bridge     : 116,599 crop records reach a polygon | 66,363 polygons labelled
gain            : +12,842 records, +8,392 polygons
records in 2015+: catastro=36  full=40


<Figure size 1200x400 with 1 Axes>

### Bridge verdict

The wider `grafica_tabular_Piura.dta` bridge is clearly **better on volume**: it links
**+12,842 more crop records** and **+8,392 more polygons** (≈57,971 → 66,363 labelled parcels),
because it simply contains more `COD_PREDIO ↔ CodigoSSET` pairs. **Use it as the bridge** — this is
what notebooks 02/03 already switched to.

But it does **not** change the temporal story: the linked records are still overwhelmingly
**1998–2007**, and 2015+ only creeps from 36 to **40 records** across all of Piura. So the Section 8
statement *"the linked crop years run ~1998–2007"* **still stands** — the better bridge buys more
parcels *within the same era*, not new years. **Sentinel-2 (2015→) remains infeasible for
year-matched training**; Landsat 5/7 is still the only sensor that covers the labelled years. The
practical constraint continues to be the crop-record dates, not the bridge.